# Kickstarter campaign data preparation

**DATA3125 · 2024–2026 launch cohort · observed outcomes through October 1, 2026 UTC**

This notebook rebuilds the cleaned data from all ten original Web Robots CSV
archives. The analysis and prediction demo can instead run directly from the
committed CSV, using `02_analysis_and_prediction.ipynb`.

The source is the [Web Robots Kickstarter dataset](https://webrobots.io/kickstarter-datasets/),
not a random sample of all Kickstarter campaigns. `sources.json` records each
original URL, byte size, and expected CSV part count. Raw ZIPs stay local and
are never modified. A snapshot date is when the scrape was taken; it is **not**
the campaign launch date. One campaign may appear in several CSV parts and
snapshots, including changing states as it finishes.

In [1]:
from pathlib import Path
import sys, json, hashlib
from zipfile import ZipFile
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'sources.json').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'sources.json').is_file(), 'Run from the project root or code/.'
sys.path.insert(0, str(ROOT / 'code'))

from kickstarter import FEATURES, EXPORT_COLUMNS, validate_cleaned
from importlib.util import spec_from_file_location, module_from_spec
spec = spec_from_file_location('download_snapshots', ROOT/'code/scripts/download_snapshots.py')
downloader = module_from_spec(spec)
spec.loader.exec_module(downloader)
sources = json.loads((ROOT/'sources.json').read_text())
snapshot_zips = [ROOT/'raw_snapshots'/item['filename'] for item in sources.values()]
for path, item in zip(snapshot_zips, sources.values()):
    if not path.is_file():
        raise FileNotFoundError('Run python code/scripts/download_snapshots.py all first: '+str(path))
    downloader.validate_archive(path, item)
display(pd.DataFrame(sources).T[['filename','bytes','csv_parts']])
january_zip = snapshot_zips[0]

,filename,bytes,csv_parts
2026-01,kickstarter_2026_01_12.zip,366117044,84
2026-02,kickstarter_2026_02_12.zip,367916874,85
2026-03,kickstarter_2026_03_12.zip,369598381,85
2026-04,kickstarter_2026_04_13.zip,371387885,86
2026-05,kickstarter_2026_05_12.zip,371972838,86
2026-06,kickstarter_2026_06_11.zip,374929166,86
2026-07,kickstarter_2026_07_24.zip,149053830,37
2026-08,kickstarter_2026_08_12.zip,337318629,79
2026-09,kickstarter_2026_09_10.zip,268088678,63
2026-10,kickstarter_2026_10_12.zip,121803899,30


## 1. Inspect January's original records

Read five rows without changing the raw fields. `id` identifies a campaign;
`launched_at`, `deadline`, and `state_changed_at` are Unix seconds. The last is
used to audit the recorded outcome, never as a model predictor. `goal` is in
local currency; `static_usd_rate` converts it to a comparable USD amount.

In [2]:
with ZipFile(january_zip) as archive:
    csv_parts = sorted(
        name for name in archive.namelist() if name.lower().endswith(".csv")
    )
    first_part = csv_parts[0]
    with archive.open(first_part) as source:
        preview = pd.read_csv(
            source, nrows=5, dtype={"id": "string"}, keep_default_na=False
        )

print(f"CSV parts in January: {len(csv_parts)}")
print(f"First part: {first_part}")
print(f"Raw columns: {len(preview.columns)}")
print(preview.columns.tolist())
fields = [
    "id", "state", "launched_at", "deadline", "state_changed_at",
    "goal", "currency", "static_usd_rate", "country",
]
display(preview[fields])

CSV parts in January: 84
First part: Kickstarter.csv
Raw columns: 42
['backers_count', 'blurb', 'category', 'converted_pledged_amount', 'country', 'country_displayable_name', 'created_at', 'creator', 'currency', 'currency_symbol', 'currency_trailing_code', 'current_currency', 'deadline', 'disable_communication', 'fx_rate', 'goal', 'id', 'is_disliked', 'is_in_post_campaign_pledging_phase', 'is_launched', 'is_liked', 'is_starrable', 'launched_at', 'location', 'name', 'percent_funded', 'photo', 'pledged', 'prelaunch_activated', 'profile', 'slug', 'source_url', 'spotlight', 'staff_pick', 'state', 'state_changed_at', 'static_usd_rate', 'urls', 'usd_exchange_rate', 'usd_pledged', 'usd_type', 'video']


,id,state,launched_at,deadline,state_changed_at,goal,currency,static_usd_rate,country
0,376968256,successful,1715094204,1717686204,1717686204,15000,USD,1,US
1,1606142212,successful,1751331437,1753934340,1753934342,50000,USD,1,US
2,1089919977,failed,1528263301,1530855301,1530855302,5000,USD,1,US
3,898133427,failed,1526064252,1528656252,1528656252,17500,USD,1,US
4,1194693022,successful,1685033518,1687625518,1687625518,9499,USD,1,US


## 2. Stack all observations before filtering

Read the same ten fields from **every** CSV part, in sorted filename order.
Append the archive date to each observation, concatenate row-wise, and
reconcile counts before dropping anything. The reference contains 2,258,203
observations across 721 CSV parts. July and October have fewer records than
neighboring snapshots, which illustrates incomplete, uneven scrape coverage.

In [3]:
from zipfile import ZipFile
import pandas as pd

raw_columns = [
    "id", "state", "launched_at", "deadline", "state_changed_at",
    "goal", "currency", "static_usd_rate", "category", "country",
]

parts = []

for path in snapshot_zips:
    snapshot_date = path.stem.removeprefix("kickstarter_").replace("_", "-")
    print(f"Reading {snapshot_date}")

    with ZipFile(path) as archive:
        csv_parts = sorted(
            name for name in archive.namelist() if name.lower().endswith(".csv")
        )

        for name in csv_parts:
            with archive.open(name) as file:
                part = pd.read_csv(
                    file,
                    usecols=raw_columns,
                    dtype={"id": "string"},
                    keep_default_na=False,
                )

            part["snapshot_date"] = snapshot_date
            parts.append(part)

raw_campaigns = pd.concat(parts, ignore_index=True)
del parts

snapshot_counts = raw_campaigns.groupby('snapshot_date').size()
assert len(raw_campaigns) == snapshot_counts.sum()
print(f"Combined shape: {raw_campaigns.shape}")
display(raw_campaigns.groupby("snapshot_date").size().rename("rows"))

Reading 2026-01-12


Reading 2026-02-12


Reading 2026-03-12


Reading 2026-04-13


Reading 2026-05-12


Reading 2026-06-11


... (2 outputs left out)

Reading 2026-09-10


Reading 2026-10-12


Combined shape: (2258203, 11)


snapshot_date
2026-01-12    266279
2026-02-12    267673
2026-03-12    268539
2026-04-13    269828
2026-05-12    270510
2026-06-11    272518
2026-07-24    112499
2026-08-12    244266
2026-09-10    194160
2026-10-12     91931
Name: rows, dtype: int64

In [4]:
ids = raw_campaigns["id"]
id_rows = raw_campaigns.loc[ids.notna() & ids.ne(""), ["id", "snapshot_date"]]

id_snapshot_pairs = id_rows.drop_duplicates()
snapshots_per_id = id_snapshot_pairs.groupby("id").size()

print(f"Unique campaign IDs: {len(snapshots_per_id):,}")
print(f"Campaign IDs seen across multiple snapshots: {snapshots_per_id.gt(1).sum():,}")
print(f"Extra rows for the same ID within a snapshot: {len(id_rows) - len(id_snapshot_pairs):,}")
# Quantify conflicting payloads within a snapshot, including model/audit fields.
# Rows with different state or goal values are not assumed interchangeable.
duplicate_pairs = raw_campaigns.duplicated(['id','snapshot_date'], keep=False)
conflict_columns = [c for c in raw_columns if c != 'id']
unique_payloads = raw_campaigns.loc[duplicate_pairs].drop_duplicates(['id','snapshot_date',*conflict_columns])
conflict_groups = unique_payloads.groupby(['id','snapshot_date']).size()
within_conflicts = int(conflict_groups.gt(1).sum())
state_conflicts = int(raw_campaigns.groupby(['id','snapshot_date'])['state'].nunique().gt(1).sum())
print(f'Within-snapshot ID groups with differing retained field values: {within_conflicts:,}')
print(f'Within-snapshot ID groups with conflicting state: {state_conflicts:,}')
display(unique_payloads.loc[unique_payloads.duplicated(['id','snapshot_date'], keep=False),
                            ['id','snapshot_date','state','goal','static_usd_rate']].head(8))
del unique_payloads

Unique campaign IDs: 236,203
Campaign IDs seen across multiple snapshots: 226,716
Extra rows for the same ID within a snapshot: 473,545


Within-snapshot ID groups with differing retained field values: 162
Within-snapshot ID groups with conflicting state: 32


,id,snapshot_date,state,goal,static_usd_rate
4625,780565037,2026-01-12,successful,2000.0,0.724257
5620,62918144,2026-01-12,successful,5000.0,0.712423
7170,1128662849,2026-01-12,successful,38943.0,0.128484
9965,1832978410,2026-01-12,live,350.0,1.177518
23540,763736584,2026-01-12,live,31499.0,0.128453
29829,24461142,2026-01-12,live,5000.0,0.669200
40595,1198098551,2026-01-12,live,2000.0,0.675995
41757,1198976852,2026-01-12,live,1000.0,0.654930


## 3. Investigate duplicates and retain one observation per campaign

The deterministic rule preserves the existing analysis: within an ID/snapshot,
keep the **first row in sorted CSV-part order**; across snapshots, keep the
latest observed row for that ID. Conflicts are counted above rather than
silently called exact duplicates. This tie-breaker is reproducible, but not
guaranteed to identify the most accurate record. It is a study limitation.

The history below shows why keeping an arbitrary early snapshot can retain a
live state after a later snapshot records success. Final state and eligibility
are evaluated **after** this deduplication; no outcome is selected merely
because it improves success rates. Latest observation means latest available
in these ten archives, not necessarily the October archive for every ID.

In [5]:
# Find the first snapshot date where each campaign was seen in each state.
first_seen = (
    raw_campaigns.loc[id_rows.index]
    .groupby(["id", "state"])["snapshot_date"]
    .min()
    .unstack()
)

# Pick one campaign that was seen live before it was seen successful.
changed = first_seen["live"] < first_seen["successful"]
example_id = first_seen.index[changed][0]

# Show that campaign's observations in date order.
history = raw_campaigns.loc[
    raw_campaigns["id"].eq(example_id), ["id", "snapshot_date", "state"]
]
display(history.drop_duplicates().sort_values("snapshot_date"))

,id,snapshot_date,state
183933,1000295722,2026-01-12,live
293797,1000295722,2026-02-12,successful
544277,1000295722,2026-03-12,successful
845618,1000295722,2026-04-13,successful
1120119,1000295722,2026-05-12,successful
1361760,1000295722,2026-06-11,successful
1710461,1000295722,2026-07-24,successful
1779109,1000295722,2026-08-12,successful
2159110,1000295722,2026-09-10,successful


In [6]:
# For the same campaign on the same date, keep the first row in CSV order.
one_per_snapshot = raw_campaigns.loc[id_rows.index].drop_duplicates(
    subset=["id", "snapshot_date"], keep="first"
)

# Sort oldest to newest, then keep the newest row for each campaign.
latest_campaigns = one_per_snapshot.sort_values("snapshot_date").drop_duplicates(
    subset="id", keep="last"
)

print(f"Rows after deduplication: {len(latest_campaigns):,}")
assert latest_campaigns["id"].is_unique

display(latest_campaigns.loc[
    latest_campaigns["id"].eq(example_id), ["id", "snapshot_date", "state"]
])

Rows after deduplication: 236,203


,id,snapshot_date,state
2159110,1000295722,2026-09-10,successful


## 4. Apply the launch, deadline, and outcome rules

Keep launches in 2024–2026. Require a deadline strictly before
**October 2, 2026 00:00 UTC** (through the end of October 1), a recorded
`successful` or `failed` state, and a state-change timestamp at or after launch
and before that same cutoff. This preserves the approved cohort. Canceled,
suspended, and live campaigns are outside the binary target.

In [7]:
# Turn Unix seconds into readable dates in UTC.
launch_seconds = pd.to_numeric(latest_campaigns["launched_at"], errors="coerce")
launch_dates = pd.to_datetime(launch_seconds, unit="s", utc=True, errors="coerce")

# Use the campaign's launch year to decide whether it belongs in our study.
launch_year = launch_dates.dt.year
in_study_years = launch_year.isin([2024, 2025, 2026])

recent_campaigns = latest_campaigns.loc[in_study_years].copy()
recent_campaigns["launch_utc"] = launch_dates.loc[in_study_years]
recent_campaigns["launch_year"] = launch_year.loc[in_study_years].astype(int)

print(f"Campaigns launched in 2024–2026: {len(recent_campaigns):,}")
display(recent_campaigns.groupby("launch_year").size().rename("campaigns"))

Campaigns launched in 2024–2026: 73,962


launch_year
2024    21646
2025    27051
2026    25265
Name: campaigns, dtype: int64

In [8]:
for source, target in [("deadline", "deadline_utc"), ("state_changed_at", "outcome_utc")]:
    seconds = pd.to_numeric(recent_campaigns[source], errors="coerce")
    recent_campaigns[target] = pd.to_datetime(
        seconds, unit="s", utc=True, errors="coerce"
    )

cutoff = pd.Timestamp("2026-10-02", tz="UTC")

deadline_ok = recent_campaigns["deadline_utc"] < cutoff
final_state = recent_campaigns["state"].isin(["successful", "failed"])
outcome_ok = (
    (recent_campaigns["outcome_utc"] < cutoff)
    & (recent_campaigns["outcome_utc"] >= recent_campaigns["launch_utc"])
)

eligible_campaigns = recent_campaigns.loc[
    deadline_ok & final_state & outcome_ok
].copy()

print(f"Eligible campaigns: {len(eligible_campaigns):,}")
display(pd.crosstab(
    eligible_campaigns["launch_year"],
    eligible_campaigns["state"],
    margins=True,
))

Eligible campaigns: 63,075


state,failed,successful,All
launch_year,,,
2024,6847,13734,20581
2025,7591,18115,25706
2026,3979,12809,16788
All,18417,44658,63075


## 5. Engineer exactly five launch-plan features

USD goal = `goal × static_usd_rate`. Duration = `(deadline − launched_at)` in
days, retaining fractional days. Launch month uses UTC. Main category comes
from the JSON parent name, falling back to the category name. Normalize
country to uppercase two-letter codes. Map successful → 1 and failed → 0.

These are campaign-plan attributes, but scraped in later snapshots: this
study does not prove each value was frozen at launch. Pledged amounts,
backers, final state, and outcome timestamps are never model inputs.

In [9]:
goal = pd.to_numeric(eligible_campaigns["goal"], errors="coerce")
usd_rate = pd.to_numeric(eligible_campaigns["static_usd_rate"], errors="coerce")

eligible_campaigns["funding_goal_usd"] = goal * usd_rate

preview_columns = ["currency", "goal", "static_usd_rate", "funding_goal_usd"]
non_usd = eligible_campaigns["currency"].ne("USD")
display(eligible_campaigns.loc[non_usd, preview_columns].head())

campaign_length = eligible_campaigns["deadline_utc"] - eligible_campaigns["launch_utc"]
eligible_campaigns["duration_days"] = campaign_length.dt.total_seconds() / 86_400

display(eligible_campaigns[["launch_utc", "deadline_utc", "duration_days"]].head())

eligible_campaigns["launch_month"] = eligible_campaigns["launch_utc"].dt.month

display(eligible_campaigns[["launch_utc", "launch_month"]].head())

,currency,goal,static_usd_rate,funding_goal_usd
162081,EUR,400.0,1.153401,461.360484
162025,GBP,500.0,1.310099,655.049580
163553,GBP,485.0,1.343301,651.500801
159412,EUR,400.0,1.086509,434.603572
164543,AUD,4200.0,0.673301,2827.863906


,launch_utc,deadline_utc,duration_days
162081,2025-11-04 17:00:06+00:00,2025-11-25 17:00:06+00:00,21.000000
162090,2025-11-04 14:55:40+00:00,2025-12-04 14:55:40+00:00,30.000000
162025,2025-11-23 10:35:31+00:00,2025-12-21 23:59:00+00:00,28.557975
162410,2025-01-28 23:00:08+00:00,2025-02-14 23:00:08+00:00,17.000000
162434,2025-01-24 15:17:15+00:00,2025-02-10 15:17:15+00:00,17.000000


,launch_utc,launch_month
162081,2025-11-04 17:00:06+00:00,11
162090,2025-11-04 14:55:40+00:00,11
162025,2025-11-23 10:35:31+00:00,11
162410,2025-01-28 23:00:08+00:00,1
162434,2025-01-24 15:17:15+00:00,1


In [10]:
import json

def extract_main_category(raw_category):
    try:
        category = json.loads(raw_category)
    except (TypeError, json.JSONDecodeError):
        return pd.NA

    if not isinstance(category, dict):
        return pd.NA

    return category.get("parent_name") or category.get("name") or pd.NA

eligible_campaigns["main_category"] = eligible_campaigns["category"].map(
    extract_main_category
)

display(eligible_campaigns["main_category"].value_counts().head(10))
print(f"Missing main categories: {eligible_campaigns['main_category'].isna().sum():,}")

eligible_campaigns["country"] = (
    eligible_campaigns["country"].astype("string").str.strip().str.upper()
)

valid_country_format = eligible_campaigns["country"].str.fullmatch(
    r"[A-Z]{2}", na=False
)

display(eligible_campaigns["country"].value_counts().head(10))
print(f"Missing or malformed country codes: {(~valid_country_format).sum():,}")

eligible_campaigns["success"] = eligible_campaigns["state"].map(
    {"successful": 1, "failed": 0}
)

display(eligible_campaigns[["state", "success"]].head())
display(eligible_campaigns["success"].value_counts(dropna=False))

main_category
Games           14390
Publishing       8256
Film & Video     6435
Technology       6262
Comics           6004
Design           5617
Art              4523
Music            3170
Fashion          2658
Food             2338
Name: count, dtype: int64

Missing main categories: 0


country
US    33948
GB     7260
CA     3292
HK     2773
DE     1924
ES     1884
AU     1758
IT     1679
FR     1557
MX     1064
Name: count, dtype: Int64

Missing or malformed country codes: 0


,state,success
162081,successful,1
162090,successful,1
162025,successful,1
162410,successful,1
162434,successful,1


success
1    44658
0    18417
Name: count, dtype: int64

## 6. Validate the export and audit calendar boundaries

Validation fails on missing/invalid features, duplicate IDs, or inconsistent
dates instead of silently changing the cohort. Outcome timestamps stay in the
small audit summary, not in the five-input model CSV. Compare deadlines and
state changes against the following January 1 for each training launch year.
This tests the limitation of interpreting a launch-year split as an as-of-date
forecast; it does not change the requested split.

In [11]:
feature_columns = FEATURES
cleaned_campaigns = eligible_campaigns[EXPORT_COLUMNS].copy()
validate_cleaned(cleaned_campaigns)
assert len(cleaned_campaigns) == 63075
assert cleaned_campaigns.groupby('launch_year').size().to_dict() == {2024:20581,2025:25706,2026:16788}
assert cleaned_campaigns['success'].value_counts().to_dict() == {1:44658,0:18417}
print('Validated 63,075 unique campaigns, all five features, dates, and target.')
boundary_rows = []
for year in [2024, 2025]:
    subset = eligible_campaigns.loc[eligible_campaigns['launch_year'].eq(year)]
    january = pd.Timestamp(f'{year+1}-01-01', tz='UTC')
    boundary_rows.append({
        'launch_year':year, 'campaigns':len(subset),
        'deadline_on_or_after_next_jan1':int(subset['deadline_utc'].ge(january).sum()),
        'outcome_on_or_after_next_jan1':int(subset['outcome_utc'].ge(january).sum()),
        'latest_recorded_outcome_utc':subset['outcome_utc'].max().isoformat(),
    })
boundary_audit = pd.DataFrame(boundary_rows)
display(boundary_audit)
audit = {
    'cutoff_exclusive_utc':cutoff.isoformat(),
    'raw_observations':len(raw_campaigns),
    'csv_parts':sum(s['csv_parts'] for s in sources.values()),
    'snapshot_rows':{str(k):int(v) for k,v in snapshot_counts.items()},
    'unique_campaign_ids':len(latest_campaigns),
    'ids_in_multiple_snapshots':int(snapshots_per_id.gt(1).sum()),
    'extra_rows_within_snapshot':len(id_rows)-len(id_snapshot_pairs),
    'within_snapshot_conflicting_payload_groups':within_conflicts,
    'within_snapshot_conflicting_state_groups':state_conflicts,
    'cohort_rows':len(cleaned_campaigns),
    'launch_year_counts':{str(k):int(v) for k,v in cleaned_campaigns.groupby('launch_year').size().items()},
    'outcome_counts':{str(k):int(v) for k,v in cleaned_campaigns.success.value_counts().items()},
    'calendar_boundaries':boundary_rows,
}

Validated 63,075 unique campaigns, all five features, dates, and target.


,launch_year,campaigns,deadline_on_or_after_next_jan1,outcome_on_or_after_next_jan1,latest_recorded_outcome_utc
0,2024,20581,1399,1399,2025-03-01T22:43:44+00:00
1,2025,25706,1506,1506,2026-03-01T13:39:45+00:00


The audit counts above use the source `state_changed_at` values. Any outcomes
at or after the following January 1 were not available at the proposed training
boundary. Even earlier state-change dates do not prove the project had been
observed by that date: these archives were scraped during 2026. A genuine
January 1 deployment simulation needs a stricter label-availability cutoff and
as-of feature snapshots. Here we report a **launch-year comparison** only.

In [12]:
# Reconcile with the existing CSV before replacing it. Compare by ID rather
# than row order; a tiny CSV float round-trip tolerance is not a cohort change.
output = ROOT/'data/kickstarter_cleaned_2024_2026.csv'
if output.is_file():
    baseline = pd.read_csv(output, dtype={'id':'string'}).set_index('id').sort_index()
    rebuilt = cleaned_campaigns.set_index('id').sort_index()
    pd.testing.assert_index_equal(baseline.index, rebuilt.index)
    for column in ['success','launch_year','launch_month','main_category','country','launch_utc','deadline_utc']:
        left, right = baseline[column], rebuilt[column]
        if column.endswith('_utc'):
            left, right = pd.to_datetime(left, utc=True), pd.to_datetime(right, utc=True)
        pd.testing.assert_series_equal(left, right, check_dtype=False)
    for column in ['funding_goal_usd','duration_days']:
        np.testing.assert_allclose(baseline[column], rebuilt[column], rtol=1e-12, atol=1e-9)
    print('Rebuild matches every baseline ID, outcome, timestamp, and model feature.')
cleaned_campaigns.to_csv(output, index=False)
audit['cleaned_csv_sha256'] = hashlib.sha256(output.read_bytes()).hexdigest()
(ROOT/'data/cohort_audit.json').write_text(json.dumps(audit, indent=2)+'\n')
print(f'Exported {len(cleaned_campaigns):,} campaigns to {output.relative_to(ROOT)}')
print('CSV SHA-256:', audit['cleaned_csv_sha256'])

Rebuild matches every baseline ID, outcome, timestamp, and model feature.


Exported 63,075 campaigns to data/kickstarter_cleaned_2024_2026.csv
CSV SHA-256: b6c852abfce2d0dced78816e348edc59b3b919485f0401e123d7764a77fb3e2e


## Data-preparation result

The exported CSV is the only dataset required by the analysis notebook.
Raw inspection, deduplication, eligibility, feature validation, and calendar
boundary evidence remain reproducible here. Consult `data/cohort_audit.json`
for the machine-readable counts and timestamp audit. The next notebook
contains the four research questions and prediction demo.